# Corpus de 10-K de las siete magníficas

Primera fase exploratoria del TFM: descargar y guardar el histórico de informes completos. La selección del último año y la investigación de noticias se harán después en otro notebook. No calcula rentabilidades, no fija una ventana bursátil y no llama a ningún LLM.

**Alcance:** todos los informes anuales originales `10-K` y su variante histórica `10-K405` que estén enumerados en los índices de submissions de EDGAR para los emisores configurados. Se recorren registros recientes y archivos históricos. Esto no garantiza recuperar documentos anteriores a EDGAR o ausentes de esos índices. Se excluyen las enmiendas `10-K/A`; no se sustituyen retrospectivamente los originales.

Alphabet incluye el emisor histórico Google, identificado por separado. Los cambios de nombre de Facebook/Meta se siguen por CIK. Los siete grupos empresariales no son siete valores bursátiles: las clases de acciones de Alphabet no se cuentan dos veces.

Se guarda el documento original y una versión de texto para lectura. El original conserva las tablas; el texto simplifica su disposición y no debe utilizarse para verificar cifras sin revisar el original. Los primeros informes históricos pueden no contener Item 1A.

**Preparación:** instalar el proyecto con sus extras de notebook y extracción, y configurar `SEC_USER_AGENT` con nombre y correo reales en el archivo local `.env`. No guardar datos de contacto en este notebook. Ejecutar desde el proyecto o desde su carpeta `notebooks`.

Fuentes: [API de EDGAR](https://www.sec.gov/search-filings/edgar-application-programming-interfaces) y [política de acceso](https://www.sec.gov/about/developer-resources).

In [1]:
from __future__ import annotations

import hashlib
import json
import sys
import time
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "src" / "data_extraction" / "sec_filings.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Abre el notebook desde el proyecto o su carpeta notebooks.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_extraction.sec_filings import (
    SecCompanyClient, SecFilingError, complete_text_filing_url, sec_user_agent,
)

user_agent = sec_user_agent(str(PROJECT_ROOT / ".env"))

pd.set_option("display.max_colwidth", 100)

## 1. Configuración del corpus

Se descarga todo el histórico enumerado por EDGAR para las siete empresas. Las descargas son secuenciales y espaciadas. Los documentos ya guardados se reutilizan; los errores quedan registrados para reintentarlos en otra ejecución.


In [2]:
COMPANIES = {
    "AAPL": {"name": "Apple", "ciks": ["0000320193"]},
    "MSFT": {"name": "Microsoft", "ciks": ["0000789019"]},
    "GOOGL": {"name": "Alphabet / Google", "ciks": ["0001652044", "0001288776"]},
    "AMZN": {"name": "Amazon", "ciks": ["0001018724"]},
    "NVDA": {"name": "NVIDIA", "ciks": ["0001045810"]},
    "META": {"name": "Meta / Facebook", "ciks": ["0001326801"]},
    "TSLA": {"name": "Tesla", "ciks": ["0001318605"]},
}
ANNUAL_FORMS = {"10-K", "10-K405"}
REQUEST_INTERVAL_SECONDS = 0.25
RUN_AT = datetime.now(timezone.utc).isoformat()
OUTPUT_DIR = PROJECT_ROOT / "data" / "mag7_10k"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

class PacedSession(requests.Session):
    """Espacia todas las peticiones SEC, incluidas las de los índices históricos."""

    def __init__(self, interval):
        super().__init__()
        self.interval = interval
        self.last_request = 0.0

    def request(self, *args, **kwargs):
        remaining = self.interval - (time.monotonic() - self.last_request)
        if remaining > 0:
            time.sleep(remaining)
        self.last_request = time.monotonic()
        return super().request(*args, **kwargs)

session = PacedSession(REQUEST_INTERVAL_SECONDS)
client = SecCompanyClient(user_agent=user_agent, session=session)

## 2. Inventario de informes disponibles

Se conserva el CIK y el nombre real del emisor, además del grupo de estudio. Un fallo al consultar un emisor se muestra explícitamente: el corpus no se considera completo.

El inventario se guarda antes de descargar documentos, de modo que un fallo de red no borra la lista de informes esperados.

In [3]:
inventory = []
inventory_errors = []
for ticker, company in COMPANIES.items():
    company_filings = {}
    for cik in company["ciks"]:
        try:
            # Sin filtro de formulario se consulta una sola vez cada página;
            # después se seleccionan ambos tipos de informe anual.
            filings = client.filing_metadata(cik)
            for filing in filings:
                if filing.form in ANNUAL_FORMS:
                    company_filings[filing.accession_number] = filing
        except (requests.RequestException, SecFilingError) as exc:
            inventory_errors.append({
                "study_ticker": ticker, "cik": cik, "error": str(exc),
            })
    ordered = sorted(
        company_filings.values(),
        key=lambda f: (f.filing_date, f.accession_number),
        reverse=True,
    )
    for filing in ordered:
        inventory.append({
            "study_ticker": ticker,
            "study_company": company["name"],
            **asdict(filing),
        })
    print(f"{ticker}: {len(ordered)} informes seleccionados")

inventory_payload = {
    "queried_at_utc": RUN_AT,
    "annual_forms": sorted(ANNUAL_FORMS),
    "inventory_errors": inventory_errors,
    "filings": inventory,
}
(OUTPUT_DIR / "inventory.json").write_text(
    json.dumps(inventory_payload, ensure_ascii=False, indent=2), encoding="utf-8"
)
if inventory_errors:
    display(pd.DataFrame(inventory_errors))
if not inventory:
    raise RuntimeError("No hay informes disponibles. Revisa los errores del inventario.")

inventory_df = pd.DataFrame(inventory)
display(inventory_df.groupby("study_ticker").agg(
    informes=("accession_number", "count"),
    primera_publicacion=("filing_date", "min"),
    ultima_publicacion=("filing_date", "max"),
))

AAPL: 32 informes seleccionados


MSFT: 33 informes seleccionados


GOOGL: 22 informes seleccionados


AMZN: 29 informes seleccionados


NVDA: 28 informes seleccionados


META: 14 informes seleccionados


TSLA: 16 informes seleccionados


,informes,primera_publicacion,ultima_publicacion
study_ticker,,,
AAPL,32,1994-12-13,2025-10-31
AMZN,29,1998-03-30,2026-02-06
GOOGL,22,2005-03-30,2026-02-05
META,14,2013-02-01,2026-01-29
MSFT,33,1994-09-27,2026-07-29
NVDA,28,1999-04-29,2026-02-25
TSLA,16,2011-03-03,2026-01-29


## 3. Descarga reanudable y texto de lectura

Cada informe conserva URL, fecha de publicación, fecha de cierre, accession, rutas relativas y huella SHA-256. Si un registro antiguo no indica documento principal o ese enlace devuelve 404, se guarda el envío completo de EDGAR y se marca como tal: puede incluir anexos.

Los errores no se convierten en resultados vacíos correctos. Una respuesta de bloqueo de la SEC detiene las siguientes descargas de esta ejecución; se mantienen el inventario y los documentos ya guardados.

In [4]:
def readable_text(document):
    """Obtiene texto de lectura; el original sigue siendo la referencia para tablas."""
    if "<html" not in document.lower():
        return document
    soup = BeautifulSoup(document, "html.parser")
    for node in soup.find_all(["script", "style", "ix:header", "ix:hidden"]):
        node.decompose()
    return "\n".join(
        line.strip() for line in soup.get_text("\n").splitlines() if line.strip()
    )

def save_manifest(records):
    payload = {**inventory_payload, "downloads": records}
    destination = OUTPUT_DIR / "manifest.json"
    temporary = destination.with_suffix(".json.part")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    temporary.replace(destination)

records = []
blocked = False
for position, item in enumerate(inventory, start=1):
    record = dict(item)
    folder = OUTPUT_DIR / item["study_ticker"] / item["accession_number"]
    folder.mkdir(parents=True, exist_ok=True)
    scope = "primary_document" if item["primary_document"] else "complete_submission"
    suffix = Path(item["primary_document"]).suffix if item["primary_document"] else ".txt"
    raw_path = folder / ("original" + (suffix or ".htm"))
    text_path = folder / "document.txt"
    source_url = item["document_url"]
    if scope == "complete_submission":
        source_url = complete_text_filing_url(item["cik"], item["accession_number"])
    fallback_path = folder / "complete_submission.txt"
    if not raw_path.exists() and fallback_path.exists():
        raw_path = fallback_path
        scope = "complete_submission"
        source_url = complete_text_filing_url(item["cik"], item["accession_number"])
    record.update({
        "source_document_url": source_url,
        "document_scope": scope,
        "raw_path": raw_path.relative_to(PROJECT_ROOT).as_posix(),
        "text_path": text_path.relative_to(PROJECT_ROOT).as_posix(),
    })
    try:
        if raw_path.exists() and raw_path.stat().st_size:
            document = raw_path.read_text(encoding="utf-8")
            record["download_status"] = "cached"
        else:
            if blocked:
                record["download_status"] = "not_attempted_after_sec_block"
                records.append(record)
                save_manifest(records)
                continue
            try:
                document = client.get_text(source_url)
            except SecFilingError as exc:
                if scope != "primary_document" or "HTTP 404" not in str(exc):
                    raise
                # Algunos índices antiguos enlazan un documento que ya no existe.
                source_url = complete_text_filing_url(item["cik"], item["accession_number"])
                document = client.get_text(source_url)
                raw_path = fallback_path
                record.update({
                    "source_document_url": source_url,
                    "document_scope": "complete_submission",
                    "raw_path": raw_path.relative_to(PROJECT_ROOT).as_posix(),
                })
            if not document.strip():
                raise ValueError("La SEC devolvió un documento vacío.")
            temporary = raw_path.with_suffix(raw_path.suffix + ".part")
            temporary.write_text(document, encoding="utf-8")
            temporary.replace(raw_path)
            record["download_status"] = "downloaded"
        text = (
            text_path.read_text(encoding="utf-8")
            if record["download_status"] == "cached" and text_path.exists()
            else readable_text(document)
        )
        if not text.strip():
            raise ValueError("No se obtuvo texto de lectura.")
        text_path.write_text(text, encoding="utf-8")
        record.update({
            "sha256": hashlib.sha256(raw_path.read_bytes()).hexdigest(),
            "word_count": len(text.split()),
            "checked_at_utc": RUN_AT,
        })
    except (requests.RequestException, SecFilingError, ValueError, OSError) as exc:
        record["download_status"] = "error"
        record["error"] = str(exc)
        if isinstance(exc, SecFilingError) and any(
            status in str(exc) for status in ("HTTP 403", "HTTP 429")
        ):
            blocked = True
    records.append(record)
    save_manifest(records)
    print(
        f"{position}/{len(inventory)} {item['study_ticker']} "
        f"{item['report_date'] or item['filing_date']}: {record['download_status']}"
    )

manifest_df = pd.DataFrame(records)
manifest_df.to_csv(OUTPUT_DIR / "manifest.csv", index=False)
display(pd.crosstab(manifest_df["study_ticker"], manifest_df["download_status"]))

1/174 AAPL 2025-09-27: cached
2/174 AAPL 2024-09-28: cached
3/174 AAPL 2023-09-30: cached
4/174 AAPL 2022-09-24: cached
5/174 AAPL 2021-09-25: cached
6/174 AAPL 2020-09-26: cached
7/174 AAPL 2019-09-28: cached
8/174 AAPL 2018-09-29: cached
9/174 AAPL 2017-09-30: cached
10/174 AAPL 2016-09-24: cached
11/174 AAPL 2015-09-26: cached
12/174 AAPL 2014-09-27: cached
13/174 AAPL 2013-09-28: cached
14/174 AAPL 2012-09-29: cached
15/174 AAPL 2011-09-24: cached
16/174 AAPL 2010-09-25: cached
17/174 AAPL 2009-09-26: cached
18/174 AAPL 2008-09-27: cached
19/174 AAPL 2007-09-29: cached
20/174 AAPL 2006-09-30: cached
21/174 AAPL 2005-09-24: cached
22/174 AAPL 2004-09-25: cached
23/174 AAPL 2003-09-27: cached
24/174 AAPL 2002-09-28: cached
25/174 AAPL 2001-09-29: cached
26/174 AAPL 2000-09-30: cached
27/174 AAPL 1999-09-25: cached
28/174 AAPL 1998-09-25: cached


29/174 AAPL 1997-09-26: cached
30/174 AAPL 1996-09-27: cached
31/174 AAPL 1995-09-29: cached
32/174 AAPL 1994-09-30: cached
33/174 MSFT 2026-06-30: cached


34/174 MSFT 2025-06-30: cached
35/174 MSFT 2024-06-30: cached
36/174 MSFT 2023-06-30: cached
37/174 MSFT 2022-06-30: cached
38/174 MSFT 2021-06-30: cached
39/174 MSFT 2020-06-30: cached
40/174 MSFT 2019-06-30: cached
41/174 MSFT 2018-06-30: cached
42/174 MSFT 2017-06-30: cached
43/174 MSFT 2016-06-30: cached
44/174 MSFT 2015-06-30: cached
45/174 MSFT 2014-06-30: cached
46/174 MSFT 2013-06-30: cached
47/174 MSFT 2012-06-30: cached
48/174 MSFT 2011-06-30: cached
49/174 MSFT 2010-06-30: cached
50/174 MSFT 2009-06-30: cached
51/174 MSFT 2008-06-30: cached
52/174 MSFT 2007-06-30: cached
53/174 MSFT 2006-06-30: cached
54/174 MSFT 2005-06-30: cached


55/174 MSFT 2004-06-30: cached
56/174 MSFT 2003-06-30: cached
57/174 MSFT 2002-06-30: cached
58/174 MSFT 2001-06-30: cached
59/174 MSFT 2000-06-30: cached
60/174 MSFT 1999-06-30: cached
61/174 MSFT 1998-06-30: cached
62/174 MSFT 1997-06-30: cached
63/174 MSFT 1996-06-30: cached
64/174 MSFT 1995-06-30: cached


65/174 MSFT 1994-06-30: cached
66/174 GOOGL 2025-12-31: cached
67/174 GOOGL 2024-12-31: cached
68/174 GOOGL 2023-12-31: cached
69/174 GOOGL 2022-12-31: cached
70/174 GOOGL 2021-12-31: cached
71/174 GOOGL 2020-12-31: cached
72/174 GOOGL 2019-12-31: cached
73/174 GOOGL 2018-12-31: cached
74/174 GOOGL 2017-12-31: cached
75/174 GOOGL 2016-12-31: cached
76/174 GOOGL 2015-12-31: cached
77/174 GOOGL 2014-12-31: cached
78/174 GOOGL 2013-12-31: cached
79/174 GOOGL 2012-12-31: cached
80/174 GOOGL 2011-12-31: cached
81/174 GOOGL 2010-12-31: cached
82/174 GOOGL 2009-12-31: cached
83/174 GOOGL 2008-12-31: cached
84/174 GOOGL 2007-12-31: cached
85/174 GOOGL 2006-12-31: cached
86/174 GOOGL 2005-12-31: cached
87/174 GOOGL 2004-12-31: cached
88/174 AMZN 2025-12-31: cached


...

123/174 NVDA 2020-01-26: cached
124/174 NVDA 2019-01-27: cached
125/174 NVDA 2018-01-28: cached
126/174 NVDA 2017-01-29: cached
127/174 NVDA 2016-01-31: cached


128/174 NVDA 2015-01-25: cached
129/174 NVDA 2014-01-26: cached
130/174 NVDA 2013-01-27: cached
131/174 NVDA 2012-01-29: cached
132/174 NVDA 2011-01-30: cached
133/174 NVDA 2010-01-31: cached
134/174 NVDA 2009-01-25: cached
135/174 NVDA 2008-03-21: cached
136/174 NVDA 2007-01-28: cached
137/174 NVDA 2006-01-29: cached
138/174 NVDA 2005-01-30: cached
139/174 NVDA 2004-01-25: cached
140/174 NVDA 2003-01-26: cached
141/174 NVDA 2002-01-27: cached
142/174 NVDA 2001-01-28: cached
143/174 NVDA 2000-01-30: cached
144/174 NVDA 1999-01-31: cached
145/174 META 2025-12-31: cached
146/174 META 2024-12-31: cached
147/174 META 2023-12-31: cached
148/174 META 2022-12-31: cached
149/174 META 2021-12-31: cached
150/174 META 2020-12-31: cached
151/174 META 2019-12-31: cached


152/174 META 2018-12-31: cached
153/174 META 2017-12-31: cached
154/174 META 2016-12-31: cached
155/174 META 2015-12-31: cached
156/174 META 2014-12-31: cached


157/174 META 2013-12-31: cached
158/174 META 2012-12-31: cached
159/174 TSLA 2025-12-31: cached
160/174 TSLA 2024-12-31: cached
161/174 TSLA 2023-12-31: cached
162/174 TSLA 2022-12-31: cached
163/174 TSLA 2021-12-31: cached
164/174 TSLA 2020-12-31: cached
165/174 TSLA 2019-12-31: cached
166/174 TSLA 2018-12-31: cached
167/174 TSLA 2017-12-31: cached
168/174 TSLA 2016-12-31: cached
169/174 TSLA 2015-12-31: cached
170/174 TSLA 2014-12-31: cached
171/174 TSLA 2013-12-31: cached
172/174 TSLA 2012-12-31: cached
173/174 TSLA 2011-12-31: cached
174/174 TSLA 2010-12-31: cached


download_status,cached
study_ticker,
AAPL,32
AMZN,29
GOOGL,22
META,14
MSFT,33
NVDA,28
TSLA,16


## Archivos generados

- `data/mag7_10k/inventory.json`: informes esperados y errores al consultar emisores.
- `data/mag7_10k/manifest.json` y `manifest.csv`: estado de cada descarga y rutas relativas.
- Una carpeta por empresa e informe, con el original y `document.txt` para lectura.

Otro notebook podrá leer el inventario para seleccionar el último informe de cada empresa. No se realiza aquí esa selección ni se buscan noticias.
